# 03 — Mean Teacher semi-supervised SegFormer-B4 ⭐ (best submission)

**This notebook produced `test_predictions_self_supervised_1.npz`, which scored 0.6174 image-wise macro Dice and placed #17 on the predictive leaderboard.**

- Sections 1–4 are the scored pipeline: augmentations → SegFormer-B4 → Mean Teacher training (student + EMA teacher, supervised CE + Dice and a softmax-MSE consistency loss on unlabeled cores) → test inference (`argmax + 1` to get class IDs 1–9).
- The sections after *Inference on Test Data* (pseudo-mask export and fine-tuning on pseudo labels) are a separate experiment: they reload a different checkpoint (`labeled_training_1.pth`) and are not part of the scored submission.

Written for Kaggle (`/kaggle/input/...` paths). See the [project README](../README.md) for the method, results and known issues.

### Imports

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Import SegFormer from Hugging Face Transformers
from transformers import SegformerForSemanticSegmentation

In [2]:
class_dict = {
    1: (0, 77, 38),
    2: (255, 255, 255),
    3: (203, 20, 20),
    4: (191, 191, 147),
    5: (133, 223, 246),
    6: (159, 157, 12),
    7: (230, 193, 156),
    8: (139, 87, 42),
    9: (200, 200, 200),
}

In [ ]:
def create_label_mask(image, class_dict):
    image_array = np.asarray(image)
    reverse_class_dict = {tuple(v): k for k, v in class_dict.items()}
    # Reshape image to 2D array of RGB tuples
    rgb_tuples = image_array.reshape(-1, 3)
    # Initialize labels array
    labels = np.zeros(rgb_tuples.shape[0], dtype=np.uint8)
    # Vectorized lookup
    for rgb, class_label in reverse_class_dict.items():
        mask = np.all(rgb_tuples == rgb, axis=1)
        labels[mask] = class_label
    return labels.reshape(image_array.shape[:2])

### 1. Define Loss Functions: Dice Loss + Combined Loss

In [4]:
def dice_loss(pred, target, eps=1e-6):
    """
    Computes the Dice loss.
    Args:
        pred: Tensor of shape (B, C, H, W) containing probabilities (after softmax).
        target: Tensor of shape (B, H, W) with ground-truth class indices.
    Returns:
        Dice loss (1 - Dice coefficient averaged over classes).
    """
    num_classes = pred.shape[1]
    # Convert target to one-hot encoding: shape (B, H, W, C) then permute to (B, C, H, W)
    target_one_hot = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()

    # Sum over batch and spatial dimensions
    dims = (0, 2, 3)
    intersection = torch.sum(pred * target_one_hot, dims)
    cardinality = torch.sum(pred + target_one_hot, dims)
    dice_score = (2. * intersection + eps) / (cardinality + eps)

    # Dice loss per class is (1 - dice_score); average over classes
    return 1 - dice_score.mean()

class CombinedLoss(nn.Module):
    """
    Combined loss: Cross-Entropy Loss + Dice Loss.
    """
    def __init__(self, weight_ce=1.0, weight_dice=1.0):
        super(CombinedLoss, self).__init__()
        self.weight_ce = weight_ce
        self.weight_dice = weight_dice
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, logits, target):
        # logits: (B, C, H, W)
        ce = self.ce_loss(logits, target)
        # Compute probabilities using softmax over class dimension
        probs = F.softmax(logits, dim=1)
        d_loss = dice_loss(probs, target)
        return self.weight_ce * ce + self.weight_dice * d_loss

### 2. Augmentation, Datasets, DataLoaders

In [ ]:
class UnlabeledDataset(Dataset):
    """
    Dataset for unlabeled images.
    Returns the image tensor and its filename.
    """
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.image_files = [f for f in os.listdir(data_dir) if f.endswith('.png')]

        self.inference_transform = transforms.Compose([
            transforms.Resize((1200, 300)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.658, 0.611, 0.486],
                                 std=[0.276, 0.287, 0.362])
                                 ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx]
        image_path = os.path.join(self.data_dir, image_file)

        try:
            image = Image.open(image_path).convert("RGB")
            (orig_w, orig_h) = image.size
            image = self.inference_transform(image)
            return image, image_file, (orig_w, orig_h)

        except FileNotFoundError as e:
            print(f"Skipping {image_file}: {e}")
            return self.__getitem__((idx + 1) % len(self))  # Skip or retry

In [ ]:
import random

class CustomAugmentation:
    def __init__(self, prob=0.5):
        self.prob = prob
        
    def __call__(self, image, mask):
        # Convert to PIL if they're tensors
        if isinstance(image, torch.Tensor):
            image = transforms.ToPILImage()(image)
        if isinstance(mask, torch.Tensor):
            mask = transforms.ToPILImage()(mask)
            
        # Random horizontal flip
        if random.random() < self.prob:
            image = transforms.functional.hflip(image)
            mask = transforms.functional.hflip(mask)
        
        # Random rotation (-10 to 10 degrees)
        if random.random() < self.prob:
            angle = random.uniform(-10, 10)
            image = transforms.functional.rotate(image, angle)
            mask = transforms.functional.rotate(mask, angle, fill=255)
            
        # Random brightness and contrast (only for image)
        if random.random() < self.prob:
            brightness_factor = random.uniform(0.8, 1.2)
            contrast_factor = random.uniform(0.8, 1.2)
            image = transforms.functional.adjust_brightness(image, brightness_factor)
            image = transforms.functional.adjust_contrast(image, contrast_factor)
            
        return image, mask

class AugmentedLabeledDataset(Dataset):
    """
    Dataset for labeled images with enhanced augmentation.
    """
    def __init__(self, data_dir, augment=True):
        self.data_dir = data_dir
        self.augment = augment
        
        # List image files that are not masks
        self.image_files = [f for f in os.listdir(data_dir) if f.endswith('_img.png')]
        
        # Define augmentations
        self.custom_aug = CustomAugmentation(prob=0.4)
        
        # Base transforms that are always applied
        self.image_transform = transforms.Compose([
            transforms.Resize((1200, 300)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.658, 0.611, 0.486],
                              std=[0.276, 0.287, 0.362])
        ])
        
        self.mask_transform = transforms.Compose([
            transforms.Resize((300, 75), interpolation=Image.NEAREST)
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx]
        image_path = os.path.join(self.data_dir, image_file)
        mask_file = image_file.replace('_img.png', '_lab.png')
        mask_path = os.path.join(self.data_dir, mask_file)

        try:
            # Load images
            image = Image.open(image_path).convert("RGB")
            mask = Image.open(mask_path)
            
            # Apply augmentations if enabled
            if self.augment:
                image, mask = self.custom_aug(image, mask)
            
            # Apply base transforms
            image = self.image_transform(image)
            mask = self.mask_transform(mask)
            label_mask = create_label_mask(mask, class_dict).astype(np.int64)

            return image, label_mask

        except FileNotFoundError as e:
            print(f"Skipping {image_file}: {e}")
            return self.__getitem__((idx + 1) % len(self))  # Skip or retry

In [ ]:
labeled_data_dir = os.path.join('/kaggle', 'input', 'core-segmentations', 'train', 'train')
unlabeled_data_dir = os.path.join('/kaggle', 'input', 'core-segmentations', 'train_unlabeled')

labeled_dataset = AugmentedLabeledDataset(labeled_data_dir, augment=True)
labeled_loader = DataLoader(labeled_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)

unlabeled_dataset = UnlabeledDataset(unlabeled_data_dir)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=1, shuffle=False)

### 3. Semi-Supervised Model (Mean Teacher Student) Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set the number of classes (adjust according to your dataset)
NUM_CLASSES = 9

# Load a pretrained SegFormer model and adjust the number of labels
model_name = "nvidia/segformer-b4-finetuned-ade-512-512"
model = SegformerForSemanticSegmentation.from_pretrained(model_name, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)
model.to(device)

# Instantiate the combined loss and the optimizer
criterion = CombinedLoss(weight_ce=1.0, weight_dice=1.0)

In [ ]:
from copy import deepcopy

class MeanTeacher:
    """
    Mean Teacher model for semi-supervised learning.
    Maintains an exponential moving average of model weights.
    """
    def __init__(self, student_model, ema_decay=0.999):
        self.student = student_model
        self.teacher = deepcopy(student_model)
        self.ema_decay = ema_decay
        
        # Disable gradients for teacher model
        for param in self.teacher.parameters():
            param.detach_()
    
    @torch.no_grad()
    def update_teacher(self):
        """Update teacher weights using exponential moving average"""
        for teacher_param, student_param in zip(self.teacher.parameters(), 
                                              self.student.parameters()):
            teacher_param.data.mul_(self.ema_decay).add_(
                student_param.data, alpha=1 - self.ema_decay)

class ConsistencyLoss(torch.nn.Module):
    """
    Implements consistency regularization loss between student and teacher predictions
    """
    def __init__(self, temperature=1.0):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, student_logits, teacher_logits):
        """
        Compute consistency loss between student and teacher predictions
        """
        student_probs = F.softmax(student_logits / self.temperature, dim=1)
        teacher_probs = F.softmax(teacher_logits / self.temperature, dim=1)
        
        return F.mse_loss(student_probs, teacher_probs.detach())

class ConfidenceBasedPseudoLabeling:
    """
    Implements confidence-based pseudo-labeling strategy
    """
    def __init__(self, confidence_threshold=0.70, warmup_epochs=5):
        self.confidence_threshold = confidence_threshold
        self.warmup_epochs = warmup_epochs
        
    def generate_pseudo_labels(self, model, unlabeled_loader, device, current_epoch):
        """
        Generate pseudo-labels for unlabeled data based on confidence threshold
        """
        model.eval()
        pseudo_labels = {}
        
        # Adjust threshold based on training progress
        if current_epoch < self.warmup_epochs:
            current_threshold = self.confidence_threshold * (current_epoch + 1) / self.warmup_epochs
        else:
            current_threshold = self.confidence_threshold
            
        with torch.no_grad():
            for images, image_files, _ in unlabeled_loader:
                images = images.to(device)
                outputs = model(pixel_values=images).logits
                probs = F.softmax(outputs, dim=1)
                
                # Get max probability and corresponding class
                confidence, predictions = torch.max(probs, dim=1)
                
                # Create confidence mask
                confidence_mask = confidence > current_threshold
                
                # Store predictions and confidence mask
                for idx, image_file in enumerate(image_files):
                    pseudo_labels[image_file] = {
                        'prediction': predictions[idx].cpu().numpy(),
                        'confidence_mask': confidence_mask[idx].cpu().numpy()
                    }
                    
        return pseudo_labels

In [ ]:
def train_semi_supervised(labeled_loader, unlabeled_loader, model, num_epochs, device, 
                         base_criterion, learning_rate=1e-4):
    """
    Implementation of semi-supervised training loop with all components
    """
    # Initialize components
    mean_teacher = MeanTeacher(model)
    consistency_loss = ConsistencyLoss()
    pseudo_labeler = ConfidenceBasedPseudoLabeling()
    
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2, eta_min=1e-6
    )
    
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        # Generate pseudo-labels for unlabeled data
        pseudo_labels = pseudo_labeler.generate_pseudo_labels(
            mean_teacher.teacher, unlabeled_loader, device, epoch
        )
        
        # visualize pseudo labels
        if (epoch + 1) % 5 == 0:
            hist = visualize_confidence_distribution(pseudo_labels)
            print(compute_pseudo_label_statistics(pseudo_labels))
            plt.hist(hist, bins='auto')
            plt.title("Histogram with 20 bins")
            plt.show()

        # Combine labeled and pseudo-labeled data
        for (labeled_imgs, labeled_masks), (unlabeled_imgs, _, _) in zip(
            labeled_loader, unlabeled_loader):
            
            # Move data to device
            labeled_imgs = labeled_imgs.to(device)
            labeled_masks = labeled_masks.to(device)
            unlabeled_imgs = unlabeled_imgs.to(device)
            
            # Forward passes
            optimizer.zero_grad()
            
            # Supervised loss on labeled data
            labeled_outputs = model(pixel_values=labeled_imgs).logits
            supervised_loss = base_criterion(labeled_outputs, labeled_masks)
            
            # Teacher prediction for unlabeled data
            with torch.no_grad():
                teacher_outputs = mean_teacher.teacher(
                    pixel_values=unlabeled_imgs).logits
            
            # Student prediction for unlabeled data
            student_outputs = model(pixel_values=unlabeled_imgs).logits
            
            # Consistency loss
            cons_loss = consistency_loss(student_outputs, teacher_outputs)
            
            # Combine losses with warm-up weights
            consistency_weight = min(epoch / 10, 1.0) * 0.5
            total_loss = supervised_loss + consistency_weight * cons_loss
            
            # Backward pass and optimization
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            # Update teacher model
            mean_teacher.update_teacher()
            
            running_loss += total_loss.item()
        
        # Update learning rate
        scheduler.step()
        
        # Print epoch statistics
        avg_loss = running_loss / len(labeled_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")
        
    # Save checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'teacher_state_dict': mean_teacher.teacher.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': avg_loss
        }, f'semi-supervised-trained_1.pth')

# Additional helper functions for monitoring and visualization
def visualize_confidence_distribution(pseudo_labels):
    """
    Visualize the distribution of confidence scores in pseudo-labels
    """
    confidences = []
    for data in pseudo_labels.values():
        mask = data['confidence_mask']
        confidences.extend(mask.flatten())
    
    return np.histogram(confidences, bins=20, range=(0, 1))

def compute_pseudo_label_statistics(pseudo_labels):
    """
    Compute statistics about pseudo-labels
    """
    total_pixels = 0
    confident_pixels = 0
    class_distribution = {}
    
    for data in pseudo_labels.values():
        mask = data['confidence_mask']
        predictions = data['prediction']
        
        total_pixels += mask.size
        confident_pixels += mask.sum()
        
        # Count class distribution for confident predictions
        unique, counts = np.unique(predictions[mask], return_counts=True)
        for cls, count in zip(unique, counts):
            class_distribution[cls] = class_distribution.get(cls, 0) + count
    
    return {
        'confidence_ratio': confident_pixels / total_pixels,
        'class_distribution': class_distribution
    }

In [ ]:
# Train using semi-supervised approach
train_semi_supervised(
    labeled_loader=labeled_loader,
    unlabeled_loader=unlabeled_loader,
    model=model,
    num_epochs=20,
    device=device,
    base_criterion=criterion
)

### 4. Inference on Test Data

In [ ]:
test_data_dir = os.path.join('/kaggle', 'input', 'core-segmentations', 'test', 'test')

test_dataset = UnlabeledDataset(test_data_dir)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
import torch.nn.functional as F

model.eval()
image_names = []
predicted_masks = []  # Dictionary to store results for each image

output_file = "test_predictions_self_supervised_1.npz"

with torch.no_grad():
    for image, image_file, image_size in test_loader:
        image = image.to(device)
        logits = model(pixel_values=image).logits  # (1, NUM_CLASSES, H, W)
        # Converting back to the original shape
        logits_upsampled = F.interpolate(
            logits, size=(image_size[1], image_size[0]), mode="bilinear", align_corners=False)
        # Convert logits to probabilities
        probs = F.softmax(logits_upsampled, dim=1)  # (1, NUM_CLASSES, H, W)

        # For each pixel, get the predicted class (argmax) and the corresponding confidence (max probability)
        confidence, pseudo_mask = torch.max(probs, dim=1)  # Both tensors of shape (1, H, W)
        pseudo_mask = pseudo_mask.squeeze(0).cpu().numpy().astype(np.uint8) + 1

        # Store using the image filename as key
        predicted_masks.append(pseudo_mask)
        image_names.append(image_file[0].replace("_img.png", "_lab"))

with open(output_file, 'wb') as f:  # Use 'wb' for binary write mode
    np.savez_compressed(f, **dict(zip(image_names, predicted_masks))) #Use np.savez for uncompressed

print(f"Inference complete! Masks saved to {output_file}")

Inference complete! Masks saved to test_predictions_self_supervised_1.npz


### Inference on Unlabeled Data: Generate Pseudo Masks with Confidences

In [ ]:
import torch.nn.functional as F

model.eval()
pseudo_masks = {}  # Dictionary to store results for each image

with torch.no_grad():
    for image, image_file, image_size in unlabeled_loader:
        image = image.to(device)

        logits = model(pixel_values=image).logits  # (1, NUM_CLASSES, H, W)
        # Converting back to the original shape
        logits_upsampled = F.interpolate(
            logits, size=(image_size[1], image_size[0]), mode="bilinear", align_corners=False)
        # Convert logits to probabilities
        probs = F.softmax(logits_upsampled, dim=1)  # (1, NUM_CLASSES, H, W)

        # For each pixel, get the predicted class (argmax) and the corresponding confidence (max probability)
        confidence, pseudo_mask = torch.max(probs, dim=1)  # Both tensors of shape (1, H, W)
        pseudo_mask = pseudo_mask.squeeze(0).cpu().numpy().astype(np.uint8) + 1

        # Store using the image filename as key
        pseudo_masks[image_file[0]] = {'mask': pseudo_mask}

# Optionally, save the pseudo masks and confidence maps to disk.
output_dir = os.path.join('data', 'pseudo_labels')
os.makedirs(output_dir, exist_ok=True)

for filename, data_dict in pseudo_masks.items():
    mask = data_dict['mask']

    # Save pseudo mask as a PNG image
    mask_img = Image.fromarray(mask)
    mask_filename = filename.replace('.png', '_pseudo_mask.png')
    mask_img.save(os.path.join(output_dir, mask_filename))

print("Pseudo labeling complete. Pseudo masks and confidence maps saved in:", output_dir)

### Pseudo Label Dataset

In [ ]:
import os
from torch.utils.data import Dataset
from PIL import Image
import torch

class PseudoLabeledDataset(Dataset):
    """
    Dataset for pseudo-labeled data.
    Assumes:
      - Unlabeled images are stored in `image_dir` (e.g., data/train_unlabeled)
      - Corresponding pseudo masks are stored in `pseudo_label_dir`
        with the naming convention: <image_basename> + '_pseudo_mask.png'
    """
    def __init__(self, image_dir, pseudo_label_dir, image_transform=None, mask_transform=None):
        self.image_dir = image_dir
        self.pseudo_label_dir = pseudo_label_dir
        self.image_transform = image_transform
        self.mask_transform = mask_transform
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith('.png')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_file = self.image_files[idx]
        image_path = os.path.join(self.image_dir, image_file)

        base, ext = os.path.splitext(image_file)
        pseudo_mask_file = base + '_pseudo_mask' + ext
        pseudo_mask_path = os.path.join(self.pseudo_label_dir, pseudo_mask_file)

        # Load the image and pseudo mask
        image = Image.open(image_path).convert("RGB")
        pseudo_mask = Image.open(pseudo_mask_path)

        if self.image_transform:
            image = self.image_transform(image)
        if self.mask_transform:
            pseudo_mask = self.mask_transform(pseudo_mask)
            pseudo_mask = np.array(pseudo_mask, dtype=np.int8).squeeze() - 1
        return image, pseudo_mask

### DataLoader for Pseudo Labeled Dataset

In [ ]:
from torchvision import transforms

train_image_transform = transforms.Compose([
    transforms.Resize((1200, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.658, 0.611, 0.486],
                         std=[0.276, 0.287, 0.362])
])

train_mask_transform = transforms.Compose([
    transforms.Resize((300, 75), interpolation=Image.NEAREST)
])

# Define the directories for unlabeled images and pseudo labels
pseudo_image_dir = os.path.join('/kaggle', 'input', 'core-segmentations', 'train_unlabeled')
pseudo_label_dir = os.path.join('/kaggle', 'input', 'core-segmentations', 'pseudo_labels', 'pseudo_labels')

# Instantiate the pseudo-labeled dataset and its DataLoader
from torch.utils.data import DataLoader

pseudo_dataset = PseudoLabeledDataset(
    image_dir=pseudo_image_dir,
    pseudo_label_dir=pseudo_label_dir,
    image_transform=train_image_transform,
    mask_transform=train_mask_transform
)

pseudo_loader = DataLoader(pseudo_dataset, batch_size=4, shuffle=True, num_workers=2)

### Fine Tune on Pseudo Labeled Data

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set the number of classes (adjust according to your dataset)
NUM_CLASSES = 9

checkpoint = torch.load("/kaggle/input/segformer-finetuned/pytorch/default/1/labeled_training_1.pth", weights_only=False)
# Load a pretrained SegFormer model and adjust the number of labels
model_name = "nvidia/segformer-b4-finetuned-ade-512-512"
model = SegformerForSemanticSegmentation.from_pretrained(model_name, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)
model.load_state_dict(checkpoint["model_state"])
model.to(device)

# Instantiate the combined loss and the optimizer
criterion = CombinedLoss(weight_ce=1.0, weight_dice=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state"])

start_epoch = checkpoint["epoch"] + 1  # Resume from the next epoch

<ipython-input-7-907200a3192f>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("/kaggle/input/segformer-finetuned/pytorch/default/1/labeled_training

config.json:   0%|          | 0.00/6.89k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/257M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b4-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([9, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([9]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import torch
import torch.nn.functional as F

num_epochs_finetune = 5  # Adjust the number of fine-tuning epochs as needed

model.train()  # Set model to training mode
for epoch in range(num_epochs_finetune):
    running_loss = 0.0
    for images, pseudo_masks in pseudo_loader:
        images = images.to(device)           # shape: (B, 3, H, W)
        pseudo_masks = pseudo_masks.to(device) # shape: (B, H, W)

        optimizer.zero_grad()

        # Forward pass through the model
        outputs = model(pixel_values=images).logits  # Shape: (B, NUM_CLASSES, h_out, w_out)

        # Resize the pseudo masks to match the output dimensions
        pseudo_masks_resized = F.interpolate(
            pseudo_masks.unsqueeze(1).float(), size=outputs.shape[2:], mode="nearest"
        ).squeeze(1).long()

        # Compute the loss on pseudo-labeled data
        loss = criterion(outputs, pseudo_masks_resized)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(pseudo_loader)
    print(f"Fine-tuning Epoch [{epoch+1}/{num_epochs_finetune}] Loss: {avg_loss:.4f}")

Fine-tuning Epoch [1/5] Loss: 1.3626
Fine-tuning Epoch [2/5] Loss: 1.0224
Fine-tuning Epoch [3/5] Loss: 0.8706
Fine-tuning Epoch [4/5] Loss: 0.7411
Fine-tuning Epoch [5/5] Loss: 0.6905


### Helper Code

In [ ]:
def load_arrays_from_npz(filename):
    """Loads arrays from a .npz file.

    Args:
        filename: The name of the .npz file.

    Returns:
        A dictionary where keys are the array names and values are the arrays.
    """
    data = np.load(filename)
    return dict(data)  # Convert the NpzFile object to a dictionary

# Example loading
loaded_arrays = load_arrays_from_npz("test_predictions.npz")

In [ ]:
arr1 = loaded_arrays['1_lab']
arr2 = loaded_arrays['2_lab']

In [ ]:
checkpoint = {
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict()
}
torch.save(checkpoint, "labeled_training_11.pth")

In [ ]:
checkpoint = torch.load("checkpoint.pth")
model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"] + 1  # Resume from the next epoch